# LoRA Fine-Tuning **Nemotron-3dot5-Lightning** on Text2SQL with NeMo AutoModel

> An end-to-end cookbook that takes you from a fresh checkout to a
> fine-tuned, deployable model — and explains *why* each knob is set the way it is.

Data prep, LoRA fine-tuning, and serving — using [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel).

No checkpoint conversion needed — AutoModel loads the HF weights directly. Three steps: data prep → fine-tune → serve.

**Model:** `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16` — a 30B-parameter hybrid Mamba-Transformer
Mixture-of-Experts with ~3B active parameters per token and a Multi-Token Prediction (MTP) head

**Task:** [BIRD-SQL](https://bird-bench.github.io/) text-to-SQL — turn a natural-language
question about a database into an executable SQL query. It is a great teaching task
because the output is *verifiable*: we can run the generated SQL and measure execution
accuracy, not just look at loss curves.

---

### ⚠️ Minimum hardware & storage
Set `n_devices` to the number of GPUs you have; expert parallelism follows it. Measured on 80 GB H100s. You also need about 130 GB of disk for the downloaded checkpoint, plus room for adapter outputs.

## Prerequisites

- At least one H100 80 GB (or similar) GPU node. 
- [NVIDIA-NeMo/Automodel](https://github.com/NVIDIA-NeMo/Automodel) cloned and its `.venv` built. See below
- The Nemotron-3.5-Lightning HF checkpoint available (local path or accessible HF Hub ID).
- `$HF_TOKEN` set in your environment (used to download BIRD datasets and the model checkpoints).

---
## Prereq Configuration

Edit the values below to match your environment. This is the only cell you should need to change.

**Before running this cell, make sure:**

1. The [AutoModel repo](https://github.com/NVIDIA-NeMo/Automodel) is cloned and its `.venv` built:
   ```bash
   git clone --branch <specific_branch_name> \
     https://github.com/NVIDIA-NeMo/Automodel.git ~/Automodel
   cd ~/Automodel
   uv venv && source .venv/bin/activate
   uv sync --frozen
   pip install mamba-ssm causal-conv1d --no-build-isolation
   ```
   Then launch Jupyter on the `.venv` kernel so `nemo_automodel` is importable.

2. The following environment variables are set before launching Jupyter:
   ```bash
   export AUTOMODEL_DIR=/path/to/your/Automodel     # where you cloned the repo
   export WORKSPACE_ROOT=/path/to/your/scratch      # needs ~130 GB free
   export HF_TOKEN=<your-huggingface-token>         # for gated model + BIRD dataset download
   ```

In [ ]:
import os
from pathlib import Path

n_devices = 8
max_seq_len = 2048  # Max sequence length for training (in tokens).
hf_model = "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16"  # HF Hub ID or local path.

automodel_dir = Path(os.environ.get("AUTOMODEL_DIR", Path.home() / "Automodel"))  # Cloned AutoModel repo with .venv.
workspace_root = Path(os.environ.get("WORKSPACE_ROOT", Path.home() / "nemotron-35-lightning-workspace"))
experiment_name = "Lightning35-text2sql-lora"

dataprep_output_dir = workspace_root / "data" / "text2sql"
checkpoint_dir = workspace_root / "checkpoints" / experiment_name

%env MODEL_ID=$hf_model
%env MAX_SEQ_LEN=$max_seq_len
%env DATAPREP_OUTPUT_DIR=$dataprep_output_dir

## 2. Dataset Prep setup

Once we have the checkpoints downloaded and NeMo Automodel cloned and the overall setup in place, we need to have the following two artifacts copied into the Automodel tree. 

1. **Our two artifacts are** — 
   - The YAML recipe and the
   - `text2sql.py` dataset target — so the config's `_target_` strings resolve.

### 2.1 Copy our artifacts into the AutoModel tree

This is the key wiring step. The recipe references the dataset by its **import path**
(`nemo_automodel.components.datasets.llm.text2sql.make_text2sql_dataset`), so `text2sql.py`
has to live *inside the installed `nemo_automodel` package*. We locate the package
automatically with `importlib`.

- YAML recipe  → `automodel_dir/examples/llm_finetune/nemotron/`
- `text2sql.py` → `automodel_dir/components/datasets/llm/`

In [ ]:
import shutil

cookbook_dir = Path().cwd()
# (a) copy the selected recipe (and its sibling) into the examples tree
nemotron_cfg_dir = automodel_dir / "examples" / "llm_finetune" / "nemotron"
if nemotron_cfg_dir.exists():
    for y in ["nemotron_mtp_lightning35_hellaswag_peft.yaml"]:
        src = cookbook_dir / y
        if src.exists():
            shutil.copy2(src, nemotron_cfg_dir / y)
            print(f"copied  {y}  ->  {nemotron_cfg_dir}")
else:
    print(f"(AutoModel not installed yet) would copy YAMLs -> {nemotron_cfg_dir}")

# (b) copy text2sql.py INTO the installed nemo_automodel package
def nemo_automodel_llm_dir():
    import importlib.util
    spec = importlib.util.find_spec("nemo_automodel")
    if spec is None or not spec.submodule_search_locations:
        return None
    return Path(list(spec.submodule_search_locations)[0]) / "components" / "datasets" / "llm"

llm_dir = nemo_automodel_llm_dir()
if llm_dir and llm_dir.exists():
    shutil.copy2(cookbook_dir / "text2sql.py", llm_dir / "text2sql.py")
    print(f"copied  text2sql.py  ->  {llm_dir}")
else:
    print("(nemo_automodel not importable yet) — after `uv sync`, re-run this cell so")
    print("text2sql.py lands at <nemo_automodel>/components/datasets/llm/text2sql.py")

## 3. Create the Text2SQL dataset (BIRD-SQL → JSONL)

NeMo AutoModel's `make_text2sql_dataset` reads JSONL with two columns — **`input`** (the prompt)
and **`output`** (the gold target). This section turns the BIRD-SQL benchmark into
`training.jsonl` / `validation.jsonl` / `test.jsonl` under `DATASET_DIR`.

We wrap each example in a consistent instruction template (question + optional external-knowledge
"evidence" + optional schema). `answer_only_loss_mask: true` in the recipe means the model is
**only scored on the target it generates**, not on the prompt tokens.

**Reasoning:** Lightning35 is a reasoning model, so with `INCLUDE_REASONING=True` we also fold in
chain-of-thought traces from a reasoning-augmented BIRD mirror — the target becomes
*reasoning → `</think>` → SQL*, teaching the model to think before it answers. Plain BIRD rows
(no trace) keep an SQL-only target, so the corpus trains both behaviours.

In [ ]:
# ---- dataset-prep settings ----
BIRD_HF_DATASET      = os.environ.get("BIRD_HF_DATASET", "xu3kev/BIRD-SQL-data-train")
REASONING_HF_DATASET = os.environ.get("REASONING_HF_DATASET", "meowterspace45/bird-sql-train-with-reasoning")
# ^ Verified mirrors: columns (db_id / question / evidence / SQL [+ reasoning_trace]) match ROW_FIELDS
#   and load with a plain `load_dataset(..., split="train")`. Override either via env if you use a
#   different mirror, adjusting ROW_FIELDS to match.
VALIDATION_FRACTION = 0.05
TEST_FRACTION       = 0.02
SPLIT_SEED          = 1234
INCLUDE_EVIDENCE    = True     # include BIRD "evidence" (external knowledge) in the prompt
INCLUDE_REASONING   = True
LIMIT               = None     # e.g. 200 for a fast smoke test; None = full dataset

ROW_FIELDS = {           # map BIRD column names -> our roles (edit if your mirror differs)
    "question":  ["question"],
    "sql":       ["SQL", "query", "sql"],
    "evidence":  ["evidence"],
    "db_id":     ["db_id", "database_id"],
    "reasoning": ["reasoning_trace"],
}

PROMPT_TEMPLATE = (
    "You are an expert data analyst. Write a single SQLite SQL query that answers the "
    "question.\n\n"
    "### Database: {db_id}\n"
    "{evidence}"
    "### Question:\n{question}\n\n"
    "### SQL:\n"
)

def _pick(row, keys):
    for k in keys:
        if k in row and row[k] not in (None, ""):
            return row[k]
    return None

def row_to_example(row):
    q   = _pick(row, ROW_FIELDS["question"])
    sql = _pick(row, ROW_FIELDS["sql"])
    if not q or not sql:
        return None
    ev   = _pick(row, ROW_FIELDS["evidence"]) if INCLUDE_EVIDENCE else None
    dbid = _pick(row, ROW_FIELDS["db_id"]) or "unknown"
    rsn  = _pick(row, ROW_FIELDS["reasoning"]) if INCLUDE_REASONING else None
    ev_block = f"### Knowledge:\n{ev}\n\n" if ev else ""
    prompt = PROMPT_TEMPLATE.format(db_id=dbid, evidence=ev_block, question=q.strip())
    # Reasoning target: emit the trace, close the think block, then the SQL. Rows without a
    # reasoning_trace (e.g. plain BIRD) just get SQL — so a mixed corpus trains both behaviours.
    output = f"{rsn.strip()}\n</think>\n\n{sql.strip()}" if rsn else sql.strip()
    return {"input": prompt, "output": output, "db_id": dbid}

print("datasets:", BIRD_HF_DATASET, "+ reasoning" if INCLUDE_REASONING else "(no reasoning)",
      "| evidence:", INCLUDE_EVIDENCE, "| limit:", LIMIT)

In [ ]:
import json, random
from datasets import load_dataset

rows = list(load_dataset(BIRD_HF_DATASET, split="train"))
print(f"loaded {len(rows)} examples from {BIRD_HF_DATASET}")
if INCLUDE_REASONING:
    try:
        r = list(load_dataset(REASONING_HF_DATASET, split="train"))
        rows += r
        print(f"+ {len(r)} reasoning examples from {REASONING_HF_DATASET}")
    except Exception as e:
        print(f"(skipping reasoning source: {e})")

random.Random(SPLIT_SEED).shuffle(rows)        # shuffle before LIMIT so a smoke test samples both sources
if LIMIT:
    rows = rows[:LIMIT]

examples = [e for e in (row_to_example(r) for r in rows) if e]

n = len(examples)
n_val  = int(n * VALIDATION_FRACTION)
n_test = int(n * TEST_FRACTION)
val   = examples[:n_val]
test  = examples[n_val:n_val + n_test]
train = examples[n_val + n_test:]

def write_jsonl(path, rows, keep_dbid=False):
    with open(path, "w") as f:
        for r in rows:
            rec = {"input": r["input"], "output": r["output"]}
            if keep_dbid:
                rec["db_id"] = r["db_id"]
            f.write(json.dumps(rec) + "\n")

train_jsonl = dataprep_output_dir / "training.jsonl"
val_jsonl   = dataprep_output_dir / "validation.jsonl"
test_jsonl  = dataprep_output_dir / "test.jsonl"
write_jsonl(train_jsonl, train)
write_jsonl(val_jsonl,   val)
write_jsonl(test_jsonl,  test, keep_dbid=True)   # keep db_id for execution-accuracy eval

n_reason = sum("</think>" in e["output"] for e in examples)
print(f"train: {len(train)}  val: {len(val)}  test: {len(test)}  (total {n}; {n_reason} with reasoning)")
print("wrote:", train_jsonl, val_jsonl, test_jsonl, sep="\n  ")
print("\n--- sample prompt+target ---\n" + train[0]["input"] + train[0]["output"][:400])

## 4. Understanding the Lightning35 recipe

The Lightning35 recipe uses FSDP2 with expert parallelism (ep_size: 8) across one H100 node, the deepep MoE dispatcher, and Transformer Engine backends — the notebook substitutes your model path, dataset paths, and checkpoint directory into the `__UPPER_CASE__` tokens before launch. LoRA adapts all modules except `*.out_proj` (excluded because Mamba layers consume that weight directly via custom kernels), with rank 8 and alpha 32, and the MTP head is configured for 2 weight-tied prediction iterations reusing the checkpoint's single physical MTP layer.


## 5. LoRA Fine-tuning

The only step that needs GPUs. This cell substitutes your paths and GPU count into the YAML recipe,
then launches training via the `automodel` CLI.

> The first iteration takes 1–2 minutes while CUDA graphs are captured and the MoE warms up.
> Subsequent iterations are seconds. Do not cancel the job.

### 5.1 Launch training and Visualize training convergence

The training job writes a log to `OUTPUT_DIR/train.log`. This cell scrapes the loss values
from that log and plots them — you can run it while training is still going to check progress,
or after the job finishes for a final view.

**What to look for:**
- Loss should trend down over steps. A flat or rising curve means something is wrong
  (learning rate too high, data issue, or the job died early).
- Train and val loss should track each other roughly. A large gap means overfitting —
  reduce `max_steps` or lower the LoRA rank.
- Final train loss below ~0.5 after one epoch on the full BIRD dataset is a good sign.

In [ ]:
import re
import matplotlib.pyplot as plt

train_jsonl = dataprep_output_dir / "training.jsonl"
val_jsonl   = dataprep_output_dir / "validation.jsonl"
config_name = "nemotron_mtp_lightning35_hellaswag_peft.yaml"
log_path    = workspace_root / "train.log"

checkpoint_dir.mkdir(parents=True, exist_ok=True)

template_path = cookbook_dir / config_name
text = template_path.read_text()

subs = {
    "__PRETRAINED_MODEL_PATH__": str(hf_model),
    "__TRAIN_JSONL__":           str(train_jsonl),
    "__VAL_JSONL__":             str(val_jsonl),
    "__CHECKPOINT_DIR__":        str(checkpoint_dir),
}
for k, v in subs.items():
    text = text.replace(k, v)

resolved_cfg = workspace_root / config_name.replace(".yaml", "_resolved.yaml")
resolved_cfg.write_text(text)

remaining = [k for k in subs if k in text]
print("Resolved config :", resolved_cfg)
print("Unsubstituted   :", remaining or "none")

print(f"\nLaunch command:")
print(f"  cd {automodel_dir}")
print(f"  source .venv/bin/activate")
print(f"  automodel {resolved_cfg} --nproc-per-node {n_devices}")

# ---------------------------------------------------------------------------
# Visualize training convergence — run after training or while it's streaming
# ---------------------------------------------------------------------------
steps, losses, val_steps, val_losses = [], [], [], []
loss_re = re.compile(r"step[^0-9]*(\d+).*?loss[:=]\s*([0-9.]+)", re.I)
val_re  = re.compile(r"val[^0-9]*?(?:step[^0-9]*(\d+)).*?loss[:=]\s*([0-9.]+)", re.I)

if log_path.exists():
    for line in log_path.read_text(errors="ignore").splitlines():
        m = val_re.search(line)
        if m:
            val_steps.append(int(m.group(1))); val_losses.append(float(m.group(2))); continue
        m = loss_re.search(line)
        if m:
            steps.append(int(m.group(1))); losses.append(float(m.group(2)))

    fig, ax = plt.subplots(figsize=(8, 4))
    if steps:     ax.plot(steps, losses, label="train loss", lw=1.5)
    if val_steps: ax.plot(val_steps, val_losses, "o-", label="val loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss")
    ax.set_title("Nemotron-3.5-Lightning LoRA / Text2SQL")
    ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
    print(f"parsed {len(steps)} train points, {len(val_steps)} val points")
else:
    print(f"No log yet at {log_path}. Run the launch command above first.")


## 6. Evaluate: generate SQL & measure execution accuracy

The real test of a Text2SQL model is whether its SQL **runs and returns the right rows**. We:
1. resolve the latest LoRA adapter checkpoint,
2. load `base + adapter` and generate SQL for held-out questions (before vs after),
3. (optionally) execute predicted vs gold SQL against the BIRD SQLite databases and score
   **execution accuracy**.

### 6.1 Resolve the latest adapter checkpoint

In [ ]:
def resolve_latest_adapter(ckpt_root: Path):
    latest = ckpt_root / "LATEST"
    if latest.exists():
        target = latest.resolve()
        cand = target / "model"
        return cand if cand.exists() else target
    steps = sorted(ckpt_root.glob("*step_*"),
                   key=lambda p: int(re.findall(r"step_(\d+)", p.name)[-1]) if re.findall(r"step_(\d+)", p.name) else -1)
    if steps:
        cand = steps[-1] / "model"
        return cand if cand.exists() else steps[-1]
    return None

adapter_dir = resolve_latest_adapter(checkpoint_dir)
print("adapter dir:", adapter_dir or f"(none found under {checkpoint_dir} — train first)")

### 6.2 Before vs after — generate on held-out questions


In [ ]:
RUN_INFERENCE = False   # set True on a GPU node with the model loaded

def load_test(n=5):
    rows = [json.loads(l) for l in open(test_jsonl)]
    return rows[:n]

if RUN_INFERENCE and adapter_dir:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    tok = AutoTokenizer.from_pretrained(hf_model, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        hf_model, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval()

    def gen(prompt, m):
        ids = tok(prompt, return_tensors="pt").to(m.device)
        out = m.generate(**ids, max_new_tokens=1024, do_sample=False)
        return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

    for ex in load_test(3):
        print("Q:", ex["input"].splitlines()[-2][:120])
        print("  gold :", ex["output"])
        print("  base :", gen(ex["input"], base).strip()[:200])
        print("  lora :", gen(ex["input"], model).strip()[:200])
        print("-" * 80)
else:
    print("Set RUN_INFERENCE=True on a GPU node (and train first) to see before/after generations.")

## 7. Deploy with vLLM

For production serving, vLLM can load the **base model + LoRA adapter** without merging (swap
adapters per request), or you can **merge** the adapter into a standalone checkpoint for
adapter-free serving.

### 7.1 Serve base + LoRA adapter

In [ ]:
vllm_serve = f"""# serve the base model with LoRA enabled (adapter applied per-request)
vllm serve "{hf_model}" \\
  --enable-lora \\
  --max-lora-rank 32 \\
  --lora-modules lightning35-sql={adapter_dir} \\
  --trust-remote-code

# then query it (OpenAI-compatible API), selecting the adapter via "model":
#   curl http://localhost:8000/v1/completions -H 'Content-Type: application/json' -d '{{
#     "model": "lightning35-sql", "prompt": "<your Text2SQL prompt>", "max_tokens": 1024 }}'
"""
print(vllm_serve)

### 7.2 (Optional) Merge the adapter into a standalone checkpoint

In [ ]:
RUN_MERGE = False   # set True on a GPU node to materialize a merged checkpoint

merged_dir = checkpoint_dir / "merged"
if RUN_MERGE and adapter_dir:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    tok = AutoTokenizer.from_pretrained(str(hf_model), trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        str(hf_model), torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
    merged.save_pretrained(str(merged_dir), safe_serialization=True)
    tok.save_pretrained(str(merged_dir))
    print("merged checkpoint ->", merged_dir)
else:
    print(f"Set RUN_MERGE=True (GPU node) to merge -> {merged_dir}")
    print("Adapter-free serving then becomes:  vllm serve", merged_dir, "--trust-remote-code")